# BirdCLEF 2024 — Day 4: Pseudo-Label Generation

**Kaggle notebook**: Generates pseudo-labels on the unlabelled Western Ghats soundscapes.

**Logic (from Chapter 4.4 of the thesis)**:
1. Evaluate unlabelled soundscapes using Teacher Ensemble (Perch + BirdNET + LCC).
2. Ensemble prediction is simple arithmetic mean.
3. Apply per-bucket thresholds (Very Rare: 0.40, Rare: 0.50, Common: 0.70).
4. If `p_ens > threshold`, hard label ($y_c = 1$). Otherwise, soft label ($y_c = p_{ens}$).

## 0. Environment & Debug Flag

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG_MODE: set True when running locally (no Kaggle data).
# On Kaggle: set False → runs the full generation and writes parquet.
# ─────────────────────────────────────────────────────────────────────────────

INPUT_DIR  = Path("/kaggle/input")
OUTPUT_DIR = Path(".")

## 1. Metadata & Rarity Buckets

In [2]:
meta_df  = pd.read_csv(INPUT_DIR / "birdclef-2024/train_metadata.csv")
ALL_SPECIES = sorted(meta_df["primary_label"].unique())
N_CLASSES   = len(ALL_SPECIES)
sp_to_idx   = {sp: i for i, sp in enumerate(ALL_SPECIES)}

counts = meta_df["primary_label"].value_counts()
very_rare = [s for s in ALL_SPECIES if counts.get(s, 0) <= 9]
rare      = [s for s in ALL_SPECIES if 9 < counts.get(s, 0) <= 50]
common    = [s for s in ALL_SPECIES if counts.get(s, 0) > 50]

very_rare_idx = [sp_to_idx[s] for s in very_rare]
rare_idx      = [sp_to_idx[s] for s in rare]
common_idx    = [sp_to_idx[s] for s in common]

print(f"Buckets → Very rare: {len(very_rare)}, Rare: {len(rare)}, Common: {len(common)}")

## 2. Ensemble Inference & Threshold Logic

In [3]:
import tensorflow as tf
import torch
import timm
import librosa
from birdnetlib import Recording
from birdnetlib.analyzer import Analyzer

# 1. Load Perch
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
pb_path = next(INPUT_DIR.rglob("saved_model.pb"))
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    perch_model = tf.saved_model.load(str(pb_path.parent))
perch_fn = perch_model.signatures["serving_default"]

def perch_embed(chunk_32k: np.ndarray) -> np.ndarray:
    x = tf.constant(chunk_32k[None, :], dtype=tf.float32)
    return perch_fn(inputs=x)["embedding"].numpy()[0]
    
# Build Perch Prototypes
train_emb_df = pd.concat([pd.read_parquet(f) for f in INPUT_DIR.rglob("perch_train*.parquet")])
prototypes = np.zeros((N_CLASSES, 1536), dtype=np.float32)
for sp, grp in train_emb_df.groupby("species_code"):
    if sp in sp_to_idx:
        prototypes[sp_to_idx[sp]] = np.stack(grp["emb"].values).mean(0)
prototypes = prototypes / (np.linalg.norm(prototypes, axis=1, keepdims=True) + 1e-12)

# 2. Load BirdNET
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    birdnet_analyzer = Analyzer()
    
# 3. Load LC-CNN
lcc_ckpt = next(INPUT_DIR.rglob("lcc_10s_wg.pth"))
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    lcc_model = timm.create_model("efficientnet_b0", pretrained=False, num_classes=182, in_chans=1)
    lcc_model.load_state_dict(torch.load(lcc_ckpt, map_location="cpu"))
lcc_model.eval()

# 4. Inference on unlabeled soundscapes
unlabeled_files = list(INPUT_DIR.rglob("unlabeled_soundscapes/*.ogg"))
print(f"Found {len(unlabeled_files)} unlabeled soundscapes.")

CHUNK_SR = 32000
CHUNK_LEN = 5 * CHUNK_SR
CHUNK_LEN10 = 10 * CHUNK_SR
import torchaudio
mel_transform = torchaudio.transforms.MelSpectrogram(sample_rate=16000, n_fft=512, hop_length=160, n_mels=64, fmin=50, fmax=8000)

rows = []

for audio_path in unlabeled_files[:100]: # In production, remove [:100] limits
    y, sr = librosa.load(str(audio_path), sr=CHUNK_SR, mono=True)
    y_16k = librosa.resample(y, orig_sr=CHUNK_SR, target_sr=16000)
    
    # BirdNET analysis
    rec = Recording(birdnet_analyzer, str(audio_path), lat=10.5, lon=76.5, date=None, min_conf=0.0)
    rec.analyze()
    
    for i, start in enumerate(range(0, len(y) - CHUNK_LEN + 1, CHUNK_LEN)):
        chunk = y[start:start + CHUNK_LEN]
        
        # Perch
        emb = perch_embed(chunk)
        emb_n = emb / (np.linalg.norm(emb) + 1e-12)
        p_perch = prototypes @ emb_n
        
        # BirdNET
        bn_vec = np.zeros(N_CLASSES)
        start_s = start / CHUNK_SR
        end_s = start_s + 5.0
        for det in rec.detections:
            if start_s <= det["start_time"] and det["end_time"] <= end_s:
                sp_code = det.get("scientific_name", "").replace(" ", "_").lower()
                if sp_code in sp_to_idx:
                    bn_vec[sp_to_idx[sp_code]] = max(bn_vec[sp_to_idx[sp_code]], det["confidence"])
        p_birdnet = bn_vec
        
        # LC-CNN
        start_16k = int((start / CHUNK_SR) * 16000)
        chunk_10s = y_16k[max(0, start_16k - int(2.5*16000)):start_16k + int(7.5*16000)]
        if len(chunk_10s) < CHUNK_LEN10:
            chunk_10s = np.pad(chunk_10s, (0, CHUNK_LEN10 - len(chunk_10s)))
        
        mel = mel_transform(torch.tensor(chunk_10s).unsqueeze(0)).unsqueeze(0)
        with torch.no_grad():
            p_lcc = torch.sigmoid(lcc_model(mel)).squeeze().numpy()
            
        # Ensemble Simple Mean
        p_ens = (p_perch + p_birdnet + p_lcc) / 3.0
        
        # --- Thesis D1: Per-Bucket Threshold Logic ---
        pseudo_label = np.zeros(N_CLASSES, dtype=np.float32)
        for c in range(N_CLASSES):
            if c in very_rare_idx:
                thresh = 0.40
            elif c in rare_idx:
                thresh = 0.50
            else:
                thresh = 0.70
            
            # Hard label if above threshold, else soft target
            if p_ens[c] > thresh:
                pseudo_label[c] = 1.0
            else:
                pseudo_label[c] = p_ens[c]
        
        rows.append({
            "filename": audio_path.name,
            "offset": end_s,
            "emb": emb,
            "pseudo_label": pseudo_label
        })

pseudo_df = pd.DataFrame(rows)
pseudo_df.to_parquet(OUTPUT_DIR / "round1_pseudo_labels.parquet")
print(f"Generated {len(pseudo_df)} pseudo-labels.")

Mock round1_pseudo_labels.parquet generated.
